### `awk`
print strings
Numbered fields
auto typing of fields
SPecial variables 
Patterns
setting variables

> Exercise: change order of fields
> Exercise: change output delimiter (TSV -> CSV)
> Exercise: length of each interval in bed file
> Average length, min, max (after custom variables)
> Walkthru together exercise: given samtools depth file, convert to BED with awk, get bases with zero coverage, merge with bedtools
> GFF:
> awk to print GFF lines with 'transcript' in 3rd column (after column info)
> awk print numbered line that have transcript in 3rd column
> awk convert gff to bed (after coords system info)
>  Variant: length of each interval in GFF file (after coords system info)

## Parsing tabular data with `awk`
Invented in the 1970's, `awk` is a scripting language included in Unix-like operating systems. It specializes in one-liner programs and manipulating text files, and like most scripting languages it is also capable of various mathematical and logical operations.

In many cases, if you're parsing information from a text file, you could write a Python script... or you could do it with `awk` in a single line! 

### `awk` syntax

`awk` scripts are organized as:

`awk 'pattern { action; other action }' input.txt`

This means that `awk` reads `input_file.txt` line by line, and for each line performs both `action` and `other action`. A semi-colon (`;`) is used to de-limit separate actions.

There is also a `pattern` that can be included before an action block, which functions similarly to an `if/then` statement in other programming languages. Meaning that every time that the pattern evaulates to *true*, `awk` will execute the action in the brackets. By default, if no pattern is specified it evaulates 'true' every line, so the action will be taken every line, e.g. the following command:

In [ ]:
%%bash
awk '{print}' data_day4/example.bed | head

We don't specify a pattern to test, so this just `print`s every line of the file. We'll show how we can make use of patterns, but first we need to discuss how `awk` interprets input.

### `awk` records and fields
As mentioned, `awk` takes tabular input (either a file or piped from `stdin`) and goes through it line by line, referring to each line as a **record**.

Each record (line) is then split by column into **fields**, based on some **field separator** (i.e. *delimiter*; which is any whitespace by default). `awk` refers to each field *by number* using the syntax `$N` (dollar sign, "N"), where `N` is the specific numbered column. E.g.:

- `$0` refers to the *entire record* (entire line)
- `$1` refers to the first field (column)
- `$2` refers to the second field (column)
- Etc.

So for example:

In [ ]:
%%bash
awk '{print $1}' data_day4/example.bed | head

This command will print the first column of the BED file, and because we do not specify a pattern before the action block it will print every line.

If we want to print multiple fields, we separate them using commas:

In [ ]:
%%bash
awk '{print $4,$1,$2,$3}' data_day4/example.bed | head

This code prints the fourth column of the BED file, then prints the first three.

To be more specific, what the `,` is actually telling `awk` to do is "print the *output field separator character*"... in other words, in plain language what the above command is saying is:

```
print the 4th column -> print output field separator -> print 1st column -> print output field separator -> print 2nd column -> print output field separator -> print 3rd column
```

#### Types of `awk` variables
One important thing to know about how `awk` handles fields is how it determines what *type* of variable each field is. If you are familiar with other programming languages, often when we first define a variable we need to tell the program if the variable is *numeric* (e.g. an integer like `11`) or a *non-numeric* (e.g. some text string like "bingus"); this is referred to as *explicitly typing* a variable.

`awk` is different in that field variables are *dynamically typed*, meaning that `awk` will attempt to "figure out" the type of a variable, without the need to set type manually. For example, if a field contains only numeric characters, `awk` will assign that field as a number.

For numeric fields in `awk`, we can use all the usual mathematical operators: addition `+`, subtraction `-`, multiplication `*`, division `/`, etc. For example, if we wanted to add `1` to each end coordinate of each interval in our BED file and print the new coordinates:

In [ ]:
%%bash
awk '{print $1,$2,$3+1,$4}' data_day4/example.bed | head

> **Exercise**:
> Write a command that prints the **length** of each interval in the file `data_day4/example.bed`.

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk '{print $3-$2}' data_day4/example.bed | head

However, an important thing to remember is that while automatic typing is convenient, it can come back to bite you! For example, say we wanted to run the command to add `1` to the end coordinate of our BED file but we mixed up the columns:

In [ ]:
%%bash
awk '{print $1+1,$2,$3,$4}' data_day4/example.bed | head

We can see that even though the first column is a character string, `awk` still allows us to add a number to it and does NOT produce an error! This is because `awk` will look at *context* when typing the variable, so if it sees a numeric operator being applied to a field (`$1+1`), it will dynamically convert that field to a numeric. Don't get tripped up and *always look at your data!*

### `awk` field separators
By default, the output field separator is a single whitespace character (i.e. ` `), and the input field separator is any whitespace character (space or a tab). So for example, if we had a comma separated file (CSV file) and tried the same command that we ran earlier:

In [ ]:
%%bash
awk '{print $4,$1,$2,$3}' data_day4/example.csv | head

We can see that it does not work correctly. Can anyone explain what is going on? 

<details><summary>Solution</summary>

- `awk` reads in each line and uses a whitespace as a field separator...
- However, there are no whitespaces! So `awk` thinks there is only a single column...
- It tries to print the 4th column, which is blank so it prints nothing...
- It prints the 1st column, which is the entire line...
- It tries to print columns 2 and 3 which are also blank

</details>

This is another example of how an incorrect command will not always produce an error message, and why it is always so important to *check yourself* and *have an expectation what what your data should look like*! 

If we want `awk` to correctly process files with field separators other than the default, we need some way to alter its behavior...

#### `awk`'s special variables
`awk` has several special built-in variables that determine how data is read in an processed, as well as store information about the data. There are a number of these variables, but the ones that are most useful to us are:

- `FS`: sets the input `F`ield `S`eparator character
- `OFS`: sets the `O`utput `F`ield `S`eparator character
- `NR`: counts the total `N`umber of `R`ecords processed so far
  - I.e. it counts the *current line number* of each line in the file

We define the field separator variables in a separate action block within our `awk` script. For example, to process our CSV file and *convert* it to a TSV file:    

In [ ]:
%%bash
# the first block defines the input and output field separators
# each definition is its own action, so we separate them with a semicolon
# the second block is the print statement, which prints the fields in the desired order separated by OFS (tabs)
awk '{FS="," ; OFS="\t"} {print $4,$1,$2,$3}' data_day4/example.csv | head

> Reminder: as we discussed last time, in text processing "tabs" are their own special characters, `\t` ("backslash t")

We can see that setting `FS=","` now correctly splits the CSV into fields, and by setting the `OFS="\t"` we are outputing the fields separated by tabs.

However, you likely noticed that the first line of the CSV file is NOT being output correctly...for the first line specifically `awk` is still not properly using the `,` as the field separator. Why??

This is due to a quirk of how `awk` processes input: `awk` reads in the first line of a file, AND THEN executes the first action block of the script, i.e. the block where we define our `FS`! So the first line uses the default whitespace separator, the `FS` block is reads, and then all subsequent lines use the correct `FS=","`.

### `BEGIN` and `END` patterns
Recall that before each action block in an `awk` script we can include a **pattern** statement, which changes `awk`'s behavior to check the pattern each line and actions in the block are only executed when that pattern is *true*. If no pattern is specified the block evaluates to *true* every line and the actions will be executed for every line.

Two built-in patterns to `awk` that are extremely useful are the `BEGIN` and `END` patterns:

- `BEGIN` executes before any lines have been read
- `END` executes after there are no more lines to read

Using the `BEGIN` pattern, we can make it so the first line of a CSV file gets the correct field separator character by defining `FS` in a block with `BEGIN`:

In [ ]:
%%bash
# before any lines are read, set FS and OFS
# print columns 4,1,2,3 for every line, separated by OFS (tabs)
awk 'BEGIN{FS="," ; OFS="\t"} {print $4,$1,$2,$3}' data_day4/example.csv | head

> **Exercise**:
> Convert the BED file `data_day4/sv.bed` to CSV format. Save it as a file `data_day4/sv.csv` 

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
awk 'BEGIN{FS="\t"; OFS=","} {print $1,$2,$3,$4}' data_day4/sv.bed > data_day4/sv.csv

head data_day4/sv.csv

> **Exercise**: 
> Write an `awk` command that prints the number of lines in the file `data_day4/example.fastq`, divided by 4.
> Note: this is a quick way to count the number of entries in a FASTQ file without havign to worry about the quality score line!

In [ ]:
%%bash
## Command here

In [ ]:
%bash
#@title Solution {display-mode: "form"}

awk 'END{print NR/4}' data_day4/example.fastq

### Using custom patterns
`BEGIN` and `END` are predefined by `awk` and are super useful, but we can also define our own patterns to pair with action blocks! A pattern is a logical expression that evaluates to *true* or *false*.